<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CKIP-BERT Multi-Task Inference

Loads artifact v3 LoRA full-data ensembles and remains backward compatible with v1/v2 full checkpoints. Input is `id,data`; output is the official five-column submission schema.


In [ ]:
# Colab dependency installation. Restart the runtime if requested.
# !pip install -q transformers peft accelerate torch pandas numpy tqdm huggingface_hub safetensors


In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download, snapshot_download
from peft import PeftModel
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


# ==========================================
# CKIP-BERT artifact v1/v2/v3 inference
# ==========================================

DEFAULT_MODEL_NAME = "ckiplab/bert-base-chinese"
DEFAULT_FOLDS = [1, 2, 3, 4, 5]
DEFAULT_MAX_LEN = 512
DEFAULT_HEAD_RATIO = 0.25
DEFAULT_T1_THRESHOLD = 0.5
DEFAULT_T3_THRESHOLD = 0.5
BATCH_SIZE = 16

ID_COLUMN = "id"
TEXT_COLUMN = "data"
TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]
V3_TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}
LEGACY_TASK_CLASSES = {
    **V3_TASK_CLASSES,
    "t2": ["already", "within_2_years", "between_2_and_5_years", "longer_than_5_years"],
}
OFFICIAL_ALLOWED_VALUES = {
    "promise_status": {"No", "Yes"},
    "verification_timeline": {
        "N/A",
        "already",
        "within_2_years",
        "between_2_and_5_years",
        "more_than_5_years",
    },
    "evidence_status": {"N/A", "No", "Yes"},
    "evidence_quality": {"N/A", "Clear", "Not Clear", "Misleading"},
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def tokenize_head_tail(text, tokenizer, max_len, head_ratio):
    body_ids = tokenizer.encode(
        f"文本：{str(text)}",
        add_special_tokens=False,
        verbose=False,
    )
    max_body_len = max_len - 2
    if len(body_ids) > max_body_len:
        head_len = int(max_body_len * head_ratio)
        tail_len = max_body_len - head_len
        body_ids = body_ids[:head_len] + body_ids[-tail_len:]
    input_ids = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]
    attention_mask = [1] * len(input_ids)
    pad_len = max_len - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    return input_ids, attention_mask


class ESGInferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, head_ratio):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.head_ratio = head_ratio

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        input_ids, attention_mask = tokenize_head_tail(
            self.df.iloc[index][TEXT_COLUMN],
            self.tokenizer,
            self.max_len,
            self.head_ratio,
        )
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }


class ESGLoraMTLModel(nn.Module):
    def __init__(self, model_name, adapter_dir):
        super().__init__()
        base_model = AutoModel.from_pretrained(model_name)
        self.backbone = PeftModel.from_pretrained(
            base_model,
            adapter_dir,
            is_trainable=False,
        )
        hidden_size = base_model.config.hidden_size
        self.shared_mlp = nn.Sequential(
            nn.Linear(hidden_size * 3, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(384, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.multi_sample_dropouts = nn.ModuleList(
            [nn.Dropout(probability) for probability in [0.10, 0.20, 0.30, 0.40, 0.50]]
        )
        self.heads = nn.ModuleDict(
            {
                task: nn.Sequential(
                    nn.Linear(256, 128),
                    nn.LayerNorm(128),
                    nn.GELU(),
                    nn.Dropout(0.20),
                    nn.Linear(128, len(labels)),
                )
                for task, labels in V3_TASK_CLASSES.items()
            }
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        hidden = outputs.last_hidden_state
        token_mask = attention_mask.unsqueeze(-1).bool()
        cls_pool = hidden[:, 0, :]
        mean_pool = (hidden * token_mask).sum(dim=1) / token_mask.sum(dim=1).clamp_min(1)
        max_pool = hidden.masked_fill(~token_mask, torch.finfo(hidden.dtype).min).max(dim=1).values
        features = self.shared_mlp(
            torch.cat([cls_pool, mean_pool, max_pool], dim=-1)
        )
        return {
            task: torch.stack(
                [head(dropout(features)) for dropout in self.multi_sample_dropouts],
                dim=0,
            ).mean(dim=0)
            for task, head in self.heads.items()
        }

    def load_heads(self, path):
        state = torch.load(path, map_location=DEVICE, weights_only=True)
        self.shared_mlp.load_state_dict(state["shared_mlp"])
        self.heads.load_state_dict(state["heads"])


class LegacyESGUnifiedMTLModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.multi_sample_dropouts = nn.ModuleList(
            [nn.Dropout(probability) for probability in [0.1, 0.2, 0.3, 0.4, 0.5]]
        )
        self.t1_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
        self.t3_head = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.LayerNorm(16),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1),
        )
        self.t2_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 4),
        )
        self.t4_head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 3),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        t1_logits = torch.stack(
            [self.t1_head(dropout(cls_output)).squeeze(-1) for dropout in self.multi_sample_dropouts]
        ).mean(dim=0)
        t3_logits = torch.stack(
            [self.t3_head(dropout(cls_output)).squeeze(-1) for dropout in self.multi_sample_dropouts]
        ).mean(dim=0)
        return (
            t1_logits,
            self.t2_head(cls_output),
            t3_logits,
            self.t4_head(cls_output),
        )


def softmax_numpy(values):
    shifted = values - values.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def sigmoid_numpy(values):
    values = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-values))


def find_artifact_root(path):
    path = Path(path)
    for candidate in [path, path / "mtl_outputs"]:
        if (candidate / "mtl_inference_config.json").exists():
            return candidate
    return None


def resolve_artifact_root(repo_id=None, model_dir=None):
    if repo_id:
        config_filename = "mtl_inference_config.json"
        try:
            config_path = Path(
                hf_hub_download(
                    repo_id=repo_id,
                    filename=config_filename,
                )
            )
        except Exception:
            config_filename = "mtl_outputs/mtl_inference_config.json"
            config_path = Path(
                hf_hub_download(
                    repo_id=repo_id,
                    filename=config_filename,
                )
            )
        with open(config_path, "r", encoding="utf-8") as file:
            remote_config = json.load(file)
        version = int(remote_config.get("artifact_version", 1))
        if version == 3:
            allow_patterns = [
                config_filename,
                "mtl_thresholds.json",
                "mtl_calibration.json",
                "tokenizer/**",
                "full_seed_*/adapter/**",
                "full_seed_*/heads.pt",
                "full_seed_*/metadata.json",
                "mtl_outputs/mtl_thresholds.json",
                "mtl_outputs/mtl_calibration.json",
                "mtl_outputs/tokenizer/**",
                "mtl_outputs/full_seed_*/adapter/**",
                "mtl_outputs/full_seed_*/heads.pt",
                "mtl_outputs/full_seed_*/metadata.json",
            ]
        else:
            allow_patterns = [
                config_filename,
                "mtl_thresholds.json",
                "tokenizer/**",
                "fold_*/best_model.pth",
                "best_mtl_model_fold_*.pth",
                "mtl_outputs/mtl_thresholds.json",
                "mtl_outputs/tokenizer/**",
                "mtl_outputs/fold_*/best_model.pth",
                "mtl_outputs/best_mtl_model_fold_*.pth",
            ]
        downloaded = snapshot_download(
            repo_id=repo_id,
            allow_patterns=allow_patterns,
        )
        root = find_artifact_root(downloaded)
        return root if root is not None else Path(downloaded)
    model_dir = Path("mtl_outputs") if model_dir is None else Path(model_dir)
    root = find_artifact_root(model_dir)
    if root is None:
        raise FileNotFoundError(
            f"Could not find CKIP MTL artifacts under {model_dir}."
        )
    return root


def load_config(root):
    path = Path(root) / "mtl_inference_config.json"
    if not path.exists():
        return {
            "artifact_version": 1,
            "model_name": DEFAULT_MODEL_NAME,
            "folds": DEFAULT_FOLDS,
            "max_len": DEFAULT_MAX_LEN,
            "head_ratio": DEFAULT_HEAD_RATIO,
            "thresholds": {
                "t1_threshold": DEFAULT_T1_THRESHOLD,
                "t3_threshold": DEFAULT_T3_THRESHOLD,
            },
        }
    with open(path, "r", encoding="utf-8") as file:
        config = json.load(file)
    if int(config.get("artifact_version", 1)) not in {1, 2, 3}:
        raise ValueError(
            f"Unsupported artifact version: {config.get('artifact_version')}"
        )
    return config


def make_data_loader(test_df, tokenizer, max_len, head_ratio):
    return DataLoader(
        ESGInferenceDataset(
            test_df,
            tokenizer,
            max_len,
            head_ratio,
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=USE_AMP,
    )


def predict_v3(model, data_loader, temperatures):
    output = {task: [] for task in V3_TASK_CLASSES}
    model.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Inference"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(input_ids, attention_mask)
            for task in V3_TASK_CLASSES:
                values = logits[task].float().cpu().numpy()
                output[task].append(
                    softmax_numpy(
                        values / float(temperatures.get(task, 1.0))
                    )
                )
    return {
        task: np.concatenate(parts, axis=0)
        for task, parts in output.items()
    }


def predict_legacy(model, data_loader):
    output = {task: [] for task in LEGACY_TASK_CLASSES}
    model.eval()
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Inference"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(input_ids, attention_mask)
            output["t1"].append(
                np.column_stack(
                    [
                        1.0 - sigmoid_numpy(logits[0].float().cpu().numpy()),
                        sigmoid_numpy(logits[0].float().cpu().numpy()),
                    ]
                )
            )
            output["t2"].append(
                softmax_numpy(logits[1].float().cpu().numpy())
            )
            output["t3"].append(
                np.column_stack(
                    [
                        1.0 - sigmoid_numpy(logits[2].float().cpu().numpy()),
                        sigmoid_numpy(logits[2].float().cpu().numpy()),
                    ]
                )
            )
            output["t4"].append(
                softmax_numpy(logits[3].float().cpu().numpy())
            )
    return {
        task: np.concatenate(parts, axis=0)
        for task, parts in output.items()
    }


def choose_t4_label(probabilities, labels, misleading_config):
    if misleading_config.get("legacy_argmax", False):
        return labels[int(np.argmax(probabilities))]
    clear_index = labels.index("Clear")
    not_clear_index = labels.index("Not Clear")
    misleading_index = labels.index("Misleading")
    standard_index = (
        clear_index
        if probabilities[clear_index] >= probabilities[not_clear_index]
        else not_clear_index
    )
    probability_threshold = float(
        misleading_config.get("probability_threshold", 1.0)
    )
    margin_threshold = float(
        misleading_config.get("margin_threshold", 1.0)
    )
    if (
        probabilities[misleading_index] >= probability_threshold
        and probabilities[misleading_index] - probabilities[standard_index]
        >= margin_threshold
    ):
        return "Misleading"
    return labels[standard_index]


def route_predictions(
    test_df,
    probabilities,
    task_classes,
    t1_threshold,
    t3_threshold,
    misleading_config=None,
):
    misleading_config = misleading_config or {
        "probability_threshold": 0.0,
        "margin_threshold": 0.0,
    }
    results = []
    for index in range(len(test_df)):
        if probabilities["t1"][index, 1] < t1_threshold:
            results.append(
                {
                    ID_COLUMN: test_df.iloc[index][ID_COLUMN],
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue
        t2_label = task_classes["t2"][
            int(probabilities["t2"][index].argmax())
        ]
        if t2_label == "longer_than_5_years":
            t2_label = "more_than_5_years"
        if probabilities["t3"][index, 1] < t3_threshold:
            results.append(
                {
                    ID_COLUMN: test_df.iloc[index][ID_COLUMN],
                    "promise_status": "Yes",
                    "verification_timeline": t2_label,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue
        results.append(
            {
                ID_COLUMN: test_df.iloc[index][ID_COLUMN],
                "promise_status": "Yes",
                "verification_timeline": t2_label,
                "evidence_status": "Yes",
                "evidence_quality": choose_t4_label(
                    probabilities["t4"][index],
                    task_classes["t4"],
                    misleading_config,
                ),
            }
        )
    return pd.DataFrame(results)[[ID_COLUMN] + TARGET_COLUMNS]


def validate_input(df):
    missing = [
        column
        for column in [ID_COLUMN, TEXT_COLUMN]
        if column not in df.columns
    ]
    if missing:
        raise ValueError(f"Test CSV missing columns: {missing}")
    if df[ID_COLUMN].duplicated().any():
        raise ValueError("Test CSV contains duplicated ids.")


def validate_output(output_df, test_df):
    expected_columns = [ID_COLUMN] + TARGET_COLUMNS
    if list(output_df.columns) != expected_columns:
        raise ValueError(
            f"Unexpected output columns: {list(output_df.columns)}"
        )
    if len(output_df) != len(test_df):
        raise ValueError("Output row count differs from test input.")
    if output_df[ID_COLUMN].tolist() != test_df[ID_COLUMN].tolist():
        raise ValueError("Output id order differs from test input.")
    for column, allowed in OFFICIAL_ALLOWED_VALUES.items():
        values = set(output_df[column].fillna("N/A").astype(str))
        invalid = values - allowed
        if invalid:
            raise ValueError(
                f"Invalid values in {column}: {sorted(invalid)}"
            )
    no_promise = output_df["promise_status"].eq("No")
    if not (
        output_df.loc[
            no_promise,
            ["verification_timeline", "evidence_status", "evidence_quality"],
        ]
        .fillna("N/A")
        .eq("N/A")
        .all(axis=None)
    ):
        raise ValueError("promise_status=No routing is invalid.")
    no_evidence = (
        output_df["promise_status"].eq("Yes")
        & output_df["evidence_status"].eq("No")
    )
    if not output_df.loc[
        no_evidence,
        "evidence_quality",
    ].fillna("N/A").eq("N/A").all():
        raise ValueError("evidence_status=No routing is invalid.")


def resolve_legacy_checkpoint(root, fold, repo_id=None):
    root = Path(root)
    new_path = root / f"fold_{fold}" / "best_model.pth"
    if new_path.exists():
        return new_path
    legacy_path = root / f"best_mtl_model_fold_{fold}.pth"
    if legacy_path.exists():
        return legacy_path
    if repo_id:
        return Path(
            hf_hub_download(
                repo_id=repo_id,
                filename=f"best_mtl_model_fold_{fold}.pth",
            )
        )
    raise FileNotFoundError(f"Missing legacy checkpoint for fold {fold}.")


def ensemble_v3(root, config, data_loader):
    members = config.get("ensemble_members", [])
    if not members:
        raise ValueError("Artifact v3 has no ensemble_members.")
    temperatures = config.get("temperatures", {})
    accumulated = {
        task: None
        for task in V3_TASK_CLASSES
    }
    for member in members:
        member_dir = Path(root) / member
        print(f"Loading CKIP-BERT LoRA member {member}")
        model = ESGLoraMTLModel(
            config.get("model_name", DEFAULT_MODEL_NAME),
            member_dir / "adapter",
        ).to(DEVICE)
        model.load_heads(member_dir / "heads.pt")
        member_probabilities = predict_v3(
            model,
            data_loader,
            temperatures,
        )
        for task, values in member_probabilities.items():
            if accumulated[task] is None:
                accumulated[task] = np.zeros_like(
                    values,
                    dtype=np.float64,
                )
            accumulated[task] += values
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return {
        task: values / len(members)
        for task, values in accumulated.items()
    }


def ensemble_legacy(root, config, data_loader, repo_id=None):
    folds = [int(fold) for fold in config.get("folds", DEFAULT_FOLDS)]
    accumulated = {task: None for task in LEGACY_TASK_CLASSES}
    for fold in folds:
        print(f"Loading legacy CKIP-BERT fold {fold}")
        checkpoint_path = resolve_legacy_checkpoint(
            root,
            fold,
            repo_id=repo_id,
        )
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )
        state_dict = checkpoint.get("model_state_dict", checkpoint)
        model = LegacyESGUnifiedMTLModel(
            config.get("model_name", DEFAULT_MODEL_NAME)
        ).to(DEVICE)
        model.load_state_dict(state_dict)
        fold_probabilities = predict_legacy(model, data_loader)
        for task, values in fold_probabilities.items():
            if accumulated[task] is None:
                accumulated[task] = np.zeros_like(
                    values,
                    dtype=np.float64,
                )
            accumulated[task] += values
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return {
        task: values / len(folds)
        for task, values in accumulated.items()
    }


def ensemble_inference_and_export(
    repo_id,
    test_csv_path,
    output_csv_path="final_submission.csv",
    model_dir=None,
):
    root = resolve_artifact_root(repo_id=repo_id, model_dir=model_dir)
    config = load_config(root)
    version = int(config.get("artifact_version", 1))
    model_name = config.get("model_name", DEFAULT_MODEL_NAME)
    max_len = int(config.get("max_len", DEFAULT_MAX_LEN))
    head_ratio = float(config.get("head_ratio", DEFAULT_HEAD_RATIO))
    thresholds = config.get("thresholds", {})
    t1_threshold = float(
        thresholds.get("t1_threshold", DEFAULT_T1_THRESHOLD)
    )
    t3_threshold = float(
        thresholds.get("t3_threshold", DEFAULT_T3_THRESHOLD)
    )

    test_df = pd.read_csv(test_csv_path).reset_index(drop=True)
    validate_input(test_df)
    tokenizer_path = Path(root) / "tokenizer"
    tokenizer = AutoTokenizer.from_pretrained(
        str(tokenizer_path) if tokenizer_path.exists() else model_name,
        use_fast=True,
    )
    data_loader = make_data_loader(
        test_df,
        tokenizer,
        max_len,
        head_ratio,
    )

    if version == 3:
        probabilities = ensemble_v3(root, config, data_loader)
        task_classes = V3_TASK_CLASSES
        misleading_config = thresholds.get("misleading", {})
    else:
        probabilities = ensemble_legacy(
            root,
            config,
            data_loader,
            repo_id=repo_id,
        )
        task_classes = LEGACY_TASK_CLASSES
        misleading_config = {
            "legacy_argmax": True,
        }

    output_df = route_predictions(
        test_df,
        probabilities,
        task_classes,
        t1_threshold=t1_threshold,
        t3_threshold=t3_threshold,
        misleading_config=misleading_config,
    )
    validate_output(output_df, test_df)
    output_df.to_csv(output_csv_path, index=False)
    print(f"Exported CKIP-BERT submission to {output_csv_path}")
    print(output_df.head())
    return output_df


In [ ]:
# ==========================================
# Run inference
# ==========================================

# Local v3 or legacy artifacts:
# ensemble_inference_and_export(
#     repo_id=None,
#     test_csv_path="/content/test.csv",
#     output_csv_path="final_submission.csv",
#     model_dir="/content/mtl_outputs",
# )

# Hugging Face artifacts:
DEFAULT_REPO_ID = "maxbeettww/VeriPromise_ESG_2026_9906"
TEST_CSV_PATH = "../data/ori_data/vpesg4k_test_2000.csv"

ensemble_inference_and_export(
    repo_id=DEFAULT_REPO_ID,
    test_csv_path=TEST_CSV_PATH,
    output_csv_path="final_submission.csv",
)
